## RQ3: Table 9 - Application-level source coverage by PII type

In [1]:
import json
import os
import glob
import pandas as pd
from collections import defaultdict
import re

In [2]:
import sys
import os
from pathlib import Path

# Resolve paths relative to this notebook so execution is independent of kernel cwd.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'RQ2_t8.ipynb').exists():
    NOTEBOOK_DIR = Path(r'i:/project2026/llmagent/RQs/RQ2')
RQS_DIR = NOTEBOOK_DIR.parent

if str(RQS_DIR) not in sys.path:
    sys.path.insert(1, str(RQS_DIR))
import config


In [3]:
def parse_filename(filepath):
    """Parses a filename to extract the app ID and database name."""
    base_name = os.path.basename(filepath)
    # Support both formats:
    # 1) PII_{APP_ID}_{DB_NAME}_{TIMESTAMP}.jsonl
    # 2) PII_{APP_ID}_{DB_NAME}.jsonl
    match = re.match(r'PII_([A-Z0-9]+)_(.*?)(?:_\d{8}T\d{6}Z)?\.jsonl$', base_name)
    if match:
        app_id = match.group(1)
        db_name = match.group(2)
        return app_id, db_name
    return None, None

def load_data(path):
    """Loads PII presence data from a directory of jsonl files."""
    # Structure: {app_id: {db_name: {pii_type: has_pii_bool}}}
    data = defaultdict(lambda: defaultdict(lambda: defaultdict(bool)))
    files = glob.glob(os.path.join(path, '*.jsonl'))
    for f_path in files:
        app_id, db_name = parse_filename(f_path)
        if not app_id or not db_name:
            continue
        with open(f_path, 'r') as f:
            for line in f:
                record = json.loads(line)
                pii_type = record['PII_type']
                if len(record['PII']) > 0:
                    data[app_id][db_name][pii_type] = True
    return data

gt_data = load_data(str(RQS_DIR / config.GROUND_TRUTH_DIR))
system_data = load_data(str(RQS_DIR / config.GPT4O_RESULTS_DIR))


In [4]:
table_data = []

for app_id, app_name in config.APP_MAPPING.items():
    row = {'ID': app_id, 'Application': app_name}
    
    app_dbs_in_gt = gt_data.get(app_id, {}).keys()

    # --- Per-PII Type Calculation ---
    for pii_type in config.PII_TYPES:
        col_name = config.COLUMN_MAPPING[pii_type]
        
        # DG(a,t): set of databases for app 'a' that contain pii_type 't' in ground truth
        gt_dbs_with_pii = {db for db in app_dbs_in_gt if gt_data.get(app_id, {}).get(db, {}).get(pii_type, False)}
        
        # DS(a,t): set of databases for app 'a' that contain pii_type 't' in system output
        system_dbs_with_pii = {db for db in app_dbs_in_gt if system_data.get(app_id, {}).get(db, {}).get(pii_type, False)}
        
        gt_count = len(gt_dbs_with_pii)
        
        if gt_count == 0:
            row[col_name] = '-'
        else:
            # covered = |DG(a,t) ∩ DS(a,t)|
            covered_count = len(gt_dbs_with_pii.intersection(system_dbs_with_pii))
            row[col_name] = f"{covered_count}/{gt_count}"

    # --- All PII Calculation ---
    # Databases in GT for this app that have *any* PII type
    gt_dbs_with_any_pii = {
        db for db in app_dbs_in_gt 
        if any(gt_data.get(app_id, {}).get(db, {}).get(pt, False) for pt in config.PII_TYPES)
    }
    
    # Databases in system output for this app that have *any* PII type
    system_dbs_with_any_pii = {
        db for db in app_dbs_in_gt
        if any(system_data.get(app_id, {}).get(db, {}).get(pt, False) for pt in config.PII_TYPES)
    }

    all_gt_count = len(gt_dbs_with_any_pii)
    if all_gt_count == 0:
        row['All PII'] = '-'
    else:
        all_covered_count = len(gt_dbs_with_any_pii.intersection(system_dbs_with_any_pii))
        row['All PII'] = f"{all_covered_count}/{all_gt_count}"
        
    table_data.append(row)

In [5]:
df = pd.DataFrame(table_data)

# Reorder columns to match Table 9
final_columns = ['ID', 'Application'] + [config.COLUMN_MAPPING[pt] for pt in config.PII_TYPES] + ['All PII']
df = df[final_columns]

df = df.set_index('ID')

# Display the dataframe
df

,Application,Email,Phone,User Name,Person Name,Postal Address,All PII
ID,,,,,,,
A1,WhatsApp,-,1/2,1/2,2/2,-,2/2
A2,Snapchat,1/1,2/2,2/2,1/2,-,2/2
A3,Telegram,-,-,-,-,-,-
A4,Google Maps,1/1,1/1,1/1,1/1,-,1/1
A5,Samsung Internet,1/1,-,-,1/1,-,2/2
I1,WhatsApp (iOS),-,1/2,0/1,2/2,1/1,2/2
I2,Contacts,1/1,1/1,-,1/1,0/1,1/1
I3,Apple Messages,1/1,0/1,-,-,-,1/1
I4,Safari,-,-,-,0/2,-,0/2


In [6]:
# Optional: Save to LaTeX
latex_output = df.to_latex(index=True, caption='Application-level source coverage by PII type.', label='tab:app_level_coverage', column_format='ll' + 'c' * (len(df.columns)))
print(latex_output)

\begin{table}
\caption{Application-level source coverage by PII type.}
\label{tab:app_level_coverage}
\begin{tabular}{llccccccc}
\toprule
 & Application & Email & Phone & User Name & Person Name & Postal Address & All PII \\
ID &  &  &  &  &  &  &  \\
\midrule
A1 & WhatsApp & - & 1/2 & 1/2 & 2/2 & - & 2/2 \\
A2 & Snapchat & 1/1 & 2/2 & 2/2 & 1/2 & - & 2/2 \\
A3 & Telegram & - & - & - & - & - & - \\
A4 & Google Maps & 1/1 & 1/1 & 1/1 & 1/1 & - & 1/1 \\
A5 & Samsung Internet & 1/1 & - & - & 1/1 & - & 2/2 \\
I1 & WhatsApp (iOS) & - & 1/2 & 0/1 & 2/2 & 1/1 & 2/2 \\
I2 & Contacts & 1/1 & 1/1 & - & 1/1 & 0/1 & 1/1 \\
I3 & Apple Messages & 1/1 & 0/1 & - & - & - & 1/1 \\
I4 & Safari & - & - & - & 0/2 & - & 0/2 \\
I5 & Calendar & 1/1 & - & - & 1/1 & - & 1/1 \\
\bottomrule
\end{tabular}
\end{table}

